In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import json
from pathlib import Path

# Load Linux Libertine font
FONT_DIR = Path(__file__).parent / 'fonts' if '__file__' in dir() else Path('fonts')
for f in FONT_DIR.glob('*.otf'):
    fontManager.addfont(str(f))

FONT_NAME = 'Linux Libertine O'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set(
    context='paper',
    style='ticks',
    palette='deep',
    font=FONT_NAME,
    font_scale=1.8,
    rc={
        'mathtext.fontset': 'stix',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.labelweight': 'normal',
    }
)
print(f'Font: {FONT_NAME}, Output: {OUTPUT_DIR}')

In [ ]:
# ---- Collect experiment data ----
EXP_BASE = Path('../../data/lake/experiments')

def load_variants(exp_dir):
    """Load all variant results from an experiment directory."""
    results = []
    rj = exp_dir / 'results.json'
    seen = set()
    if rj.exists():
        with open(rj) as f:
            data = json.load(f)
        for v in data.get('variants', []):
            results.append(v)
            seen.add(v['variant_name'])
    for sub in sorted(exp_dir.iterdir()):
        vr = sub / 'variant_result.json'
        if vr.exists():
            with open(vr) as f:
                v = json.load(f)
            if v.get('variant_name', sub.name) not in seen:
                results.append(v)
    return results

# Concept ablation: tc10, tc25, tc50, tc75, tc100
concept_exp = EXP_BASE / 'concept_ablation_fetaqa_20260306_164425'
concept_variants = load_variants(concept_exp)
concept_data = []
for v in concept_variants:
    tc = v['config']['target_classes']
    s = v['llm_stats']
    concept_data.append((tc, s['total_requests'], s['total_input_tokens'], s['total_output_tokens']))
concept_data.sort()

# Query ablation: q50, q100, q200, q400 + q700, q1000
query_data = []
for qdir in ['query_ablation_fetaqa_20260306_154600', 'query_ablation_fetaqa_20260306_162558']:
    for v in load_variants(EXP_BASE / qdir):
        nq = v['config']['n_queries']
        s = v['llm_stats']
        query_data.append((nq, s['total_requests'], s['total_input_tokens'], s['total_output_tokens']))
query_data.sort()

# Iteration ablation: derive from iter10 by_caller breakdown
iter_exp = EXP_BASE / 'iter_ablation_fetaqa_20260306_153225'
with open(iter_exp / 'iter10' / 'variant_result.json') as f:
    iter10 = json.load(f)
bc = iter10['llm_stats']['by_caller']
N_ITER = 10

# Fixed cost (run once)
fixed_callers = ['branch_generate_cqs', 'global_init_node', 'final_dp_generation_node']
fixed_req = sum(bc[c]['requests'] for c in fixed_callers)
fixed_in = sum(bc[c]['input_tokens'] for c in fixed_callers)
fixed_out = sum(bc[c]['output_tokens'] for c in fixed_callers)

# Per-iteration cost
iter_callers = [c for c in bc if c not in fixed_callers]
iter_req = sum(bc[c]['requests'] for c in iter_callers) / N_ITER
iter_in = sum(bc[c]['input_tokens'] for c in iter_callers) / N_ITER
iter_out = sum(bc[c]['output_tokens'] for c in iter_callers) / N_ITER

iter_data = []
for n in [1, 3, 5, 7, 10]:
    iter_data.append((
        n,
        int(fixed_req + iter_req * n),
        int(fixed_in + iter_in * n),
        int(fixed_out + iter_out * n),
    ))

print('Concept:', concept_data)
print('Query:', query_data)
print('Iteration:', iter_data)

In [ ]:
# ---- Load quality metrics for the second row ----
import os
print(f'CWD: {os.getcwd()}')

DATA_DIR = Path(__file__).parent / 'data' if '__file__' in dir() else Path('data')
print(f'DATA_DIR: {DATA_DIR.resolve()}')
print(f'Files: {list(DATA_DIR.glob("ablation_*.json"))}')

# BM25 retrieval results (common query set)
with open(DATA_DIR / 'ablation_bm25_results.json') as f:
    bm25_all = json.load(f)
bm25 = bm25_all['variants']

# TBox coverage metrics
with open(DATA_DIR / 'ablation_concept_tbox_metrics.json') as f:
    concept_tbox = json.load(f)['variants']
with open(DATA_DIR / 'ablation_query_tbox_metrics.json') as f:
    query_tbox = json.load(f)['variants']
with open(DATA_DIR / 'ablation_iter_tbox_metrics.json') as f:
    iter_tbox = json.load(f)['variants']

# Build data tuples: (x_value, coverage, raw_r1, upo_r1)
concept_quality = []
for tc in [10, 25, 50, 75, 100]:
    label = f'tc{tc}'
    cov = concept_tbox[label]['avg_coverage_ratio']
    raw_r1 = bm25[label]['raw']['recall_1'] * 100
    upo_r1 = bm25[label]['combined']['recall_1'] * 100
    concept_quality.append((tc, cov, raw_r1, upo_r1))

query_quality = []
for q in [50, 100, 200, 400, 700, 1000]:
    label = f'q{q}'
    cov = query_tbox[label]['avg_coverage_ratio']
    raw_r1 = bm25[label]['raw']['recall_1'] * 100
    upo_r1 = bm25[label]['combined']['recall_1'] * 100
    query_quality.append((q, cov, raw_r1, upo_r1))

iter_quality = []
for n in [1, 3, 5, 7, 10]:
    label = f'iter{n}'
    cov = iter_tbox[label]['avg_coverage_ratio']
    raw_r1 = bm25[label]['raw']['recall_1'] * 100
    upo_r1 = bm25[label]['combined']['recall_1'] * 100
    iter_quality.append((n, cov, raw_r1, upo_r1))

print('Concept quality:', concept_quality)
print('Query quality:', query_quality)
print('Iter quality:', iter_quality)

In [ ]:
# ---- Scaling visualization: 3 subplots side by side (double-column) ----
COLOR_CALLS = '#3C6E8F'   # teal blue for LLM Calls
COLOR_TOKENS = '#C08552'  # warm bronze for Tokens

datasets = [
    (r'$M_c$', concept_data),
    (r'$|\mathcal{Q}|$', query_data),
    (r'$T$', iter_data),
]

# Calculate unified y-axis range for LLM Calls
all_calls = [d[1] for ds in [concept_data, query_data, iter_data] for d in ds]
calls_min, calls_max = min(all_calls), max(all_calls)
calls_margin = (calls_max - calls_min) * 0.08
calls_ylim = (calls_min - calls_margin, calls_max + calls_margin)

fig, axes = plt.subplots(1, 3, figsize=(7.09, 1.8))
ax2_list = []

for idx, (ax, (xlabel, data)) in enumerate(zip(axes, datasets)):
    xs = [d[0] for d in data]
    calls = [d[1] for d in data]
    input_tok = [d[2] / 1000 for d in data]   # in K
    output_tok = [d[3] / 1000 for d in data]  # in K

    # Left Y-axis: LLM Calls
    ln1 = ax.plot(xs, calls, color=COLOR_CALLS, marker='o',
                  markersize=4, linewidth=1.3, label='LLM Calls', zorder=3)
    ax.set_xlabel(xlabel, fontsize=9.4, labelpad=2)
    ax.set_ylim(calls_ylim)  # unified y-axis range
    # Move x-ticks to inside (top)
    ax.tick_params(axis='x', labelsize=8, pad=2, direction='in')
    
    # Left y-axis: show ticks on all subplots, but only label on first
    if idx == 0:
        ax.set_ylabel('LLM invocations', fontsize=8, color=COLOR_CALLS, labelpad=0)
        ax.tick_params(axis='y', labelcolor=COLOR_CALLS, labelsize=8, pad=1, direction='in')
    else:
        ax.tick_params(axis='y', labelleft=False, direction='in')  # hide labels but show ticks

    # Right Y-axis: Tokens (K) — shared color, solid vs dashed
    ax2 = ax.twinx()
    ax2_list.append(ax2)
    ln2 = ax2.plot(xs, input_tok, color=COLOR_TOKENS, marker='s',
                   markersize=4, linewidth=1.3, linestyle='-',
                   label='Input Tokens (K)', zorder=3)
    ln3 = ax2.plot(xs, output_tok, color=COLOR_TOKENS, marker='^',
                   markersize=4, linewidth=1.3, linestyle='--',
                   label='Output Tokens (K)', zorder=3)
    
    # Show right y-axis ticks on all subplots (different scales), no ylabel
    # Start from 0, add margin at top, don't show 0 as tick
    max_tok = max(input_tok)
    ax2.set_ylim(bottom=0, top=max_tok * 1.15)  # increased top margin
    current_ticks = ax2.get_yticks()
    ax2.set_yticks([t for t in current_ticks if t > 0 and t < max_tok * 1.05])
    ax2.tick_params(axis='y', labelcolor=COLOR_TOKENS, labelsize=8, pad=1, direction='in')

    # Light grid on left axis only
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)

    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_linewidth(0.5)
        ax2.spines[spine].set_linewidth(0.5)

# Set integer xticks for iteration subplot
axes[2].set_xticks([1, 3, 5, 7, 10])

# Legend in upper left of rightmost subplot (2 rows, 1 col)
lines = ln2 + ln3  # Only token lines in legend
labels = [l.get_label() for l in lines]
axes[2].legend(lines, labels, loc='upper left', ncol=1,
               frameon=True, fontsize=7, handlelength=1.5, handletextpad=0.3,
               fancybox=False, edgecolor='#BBBBBB', borderpad=0.3)

fig.subplots_adjust(wspace=0.18, left=0.07, right=0.92, bottom=0.20, top=0.90)
fig.savefig(OUTPUT_DIR / 'scaling_llm.pdf', bbox_inches='tight', pad_inches=0.02, dpi=300)
plt.show()
print(f'Saved to {OUTPUT_DIR / "scaling_llm.pdf"}')

In [ ]:
# ---- Combined 1x3 figure: Tokens (left) + Quality% (right) ----
from matplotlib.lines import Line2D

COLOR_IN_TOK  = '#C0392B'  # red — Input Tokens
COLOR_OUT_TOK = '#D4A017'  # golden yellow — Output Tokens
COLOR_COV     = '#6B5B95'  # royal purple — Avg. Class Utilization
COLOR_UPO     = '#3C6E8F'  # teal blue — +UPO Hit@1

cost_datasets = [
    (r'$M_c$', concept_data),
    (r'$|\mathcal{Q}|$', query_data),
    (r'$T$', iter_data),
]
quality_datasets = [
    (r'$M_c$', concept_quality),
    (r'$|\mathcal{Q}|$', query_quality),
    (r'$T$', iter_quality),
]

# Unified left y-axis (Tokens K)
all_in = [d[2] / 1000 for ds in [concept_data, query_data, iter_data] for d in ds]
all_out = [d[3] / 1000 for ds in [concept_data, query_data, iter_data] for d in ds]
tok_vals = all_in + all_out
tok_ylim = (0, max(tok_vals) * 1.12)

# Unified right y-axis (percentages)
all_cov = [d[1] * 100 for ds in [concept_quality, query_quality, iter_quality] for d in ds]
all_upo = [d[3] for ds in [concept_quality, query_quality, iter_quality] for d in ds]
pct_vals = all_cov + all_upo
pct_margin = (max(pct_vals) - min(pct_vals)) * 0.15
pct_ylim = (min(pct_vals) - pct_margin, max(pct_vals) + pct_margin)

fig, axes = plt.subplots(1, 3, figsize=(7.09, 1.5))
twin_axes = []

for idx in range(3):
    ax = axes[idx]
    xlabel, cdata = cost_datasets[idx]
    _, qdata = quality_datasets[idx]

    xs = [d[0] for d in cdata]
    in_tok = [d[2] / 1000 for d in cdata]
    out_tok = [d[3] / 1000 for d in cdata]
    covs = [d[1] * 100 for d in qdata]
    upo = [d[3] for d in qdata]

    # Left y-axis: Tokens
    ax.plot(xs, in_tok, color=COLOR_IN_TOK, marker='o', markersize=3,
            linewidth=1.1, linestyle='--', zorder=3)
    ax.plot(xs, out_tok, color=COLOR_OUT_TOK, marker='s', markersize=3,
            linewidth=1.1, linestyle='--', zorder=3)
    ax.set_ylim(tok_ylim)
    ax.set_xlabel(xlabel, fontsize=9, labelpad=0)
    ax.tick_params(axis='x', labelsize=6.5, pad=1, direction='in')
    ax.tick_params(axis='y', labelsize=6.5, pad=0.5, direction='in',
                   labelleft=(idx == 0))

    if idx == 0:
        ax.set_ylabel('Input Tokens (K)\nOutput Tokens (K)', fontsize=6.5,
                       color='#333333', labelpad=0)

    # Right y-axis: percentages
    ax2 = ax.twinx()
    twin_axes.append(ax2)
    ax2.plot(xs, covs, color=COLOR_COV, marker='^', markersize=3,
             linewidth=1.1, zorder=3)
    ax2.plot(xs, upo, color=COLOR_UPO, marker='D', markersize=2.5, linewidth=1.1,
             zorder=3)
    ax2.set_ylim(pct_ylim)
    ax2.tick_params(axis='y', labelsize=6.5, pad=0.5, direction='in',
                    labelright=(idx == 2))

    if idx == 2:
        ax2.set_ylabel('Avg. Class Util. (%)\nSaturn Hit@1 (%)', fontsize=6.5,
                        color='#333333', labelpad=0)

    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_linewidth(0.5)
        ax2.spines[spine].set_linewidth(0.5)

axes[2].set_xticks([1, 3, 5, 7, 10])

# Legend at top
legend_handles = [
    Line2D([0], [0], color=COLOR_IN_TOK, marker='o', markersize=3.5, linewidth=1.1, linestyle='--', label='Input Tokens (K)'),
    Line2D([0], [0], color=COLOR_OUT_TOK, marker='s', markersize=3.5, linewidth=1.1, linestyle='--', label='Output Tokens (K)'),
    Line2D([0], [0], color=COLOR_COV, marker='^', markersize=3.5, linewidth=1.1, label='Avg. Class Util. (%)'),
    Line2D([0], [0], color=COLOR_UPO, marker='D', markersize=3, linewidth=1.1, label='+UPO Hit@1 (%)'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=4,
           frameon=True, fontsize=6, handlelength=1.5, handletextpad=0.3,
           fancybox=False, edgecolor='#BBBBBB', borderpad=0.15,
           bbox_to_anchor=(0.5, 1.0))

fig.subplots_adjust(wspace=0.05, left=0.10, right=0.90, bottom=0.18, top=0.95)

fig.savefig(OUTPUT_DIR / 'scaling_combined.pdf', bbox_inches='tight', pad_inches=0, dpi=300)
print(f'Saved to {OUTPUT_DIR / "scaling_combined.pdf"}')